# YAICON F5-TTS comparison

Private, non-commercial research benchmark. An AI persona simulation uses the same reference and English answer as version 3. F5-TTS v1 Base is measured at 16 and 32 denoising steps.

In [ ]:
from pathlib import Path
from importlib.metadata import version
from time import perf_counter
import base64
import hashlib
import json
import os
import subprocess
import sys
import tempfile

import torch

assert torch.cuda.is_available(), "The experiment needs a free CUDA session."
print("Device:", torch.cuda.get_device_name(0), "PyTorch:", torch.__version__, flush=True)
work = Path(tempfile.mkdtemp(prefix="yaicon-f5-"))
out = Path("/kaggle/working")
os.environ.update(HF_HUB_DISABLE_TELEMETRY="1", HF_HUB_DISABLE_XET="1", USE_TF="0",
                  HF_HUB_DOWNLOAD_TIMEOUT="90", HF_HUB_DISABLE_PROGRESS_BARS="1",
                  TOKENIZERS_PARALLELISM="false", PIP_DISABLE_PIP_VERSION_CHECK="1")


In [ ]:
# Attach a private Kaggle input containing the experiment assets, then set this path.
input_path = Path("/kaggle/input/CHANGE-ME")
import shutil

for name in ("reference.wav", "reference.txt", "benchmark.json"):
    shutil.copyfile(input_path / name, work / name)


In [ ]:
constraints = work / "constraints.txt"
constraints.write_text("".join(f"{name}=={version(name)}\n" for name in ("torch", "torchvision", "torchaudio")))
print("Installing F5-TTS and retaining Kaggle's CUDA packages.", flush=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--break-system-packages", "-q",
                "-c", str(constraints), "f5-tts==1.1.22", "audioop-lts; python_version >= '3.13'"],
               timeout=360, check=True)


In [ ]:
runner = work / "benchmark-f5.py"
runner.write_text('from contextlib import redirect_stdout\nfrom importlib.metadata import version\nfrom pathlib import Path\nfrom time import perf_counter\nimport hashlib\nimport io\nimport json\nimport statistics\nimport sys\n\nimport numpy as np\nimport soundfile as sf\nimport torch\nfrom f5_tts.api import F5TTS\nfrom huggingface_hub import hf_hub_download\n\nwork, out = map(Path, sys.argv[1:])\nfixture = json.loads((work / "benchmark.json").read_text())\nreference = work / "reference.wav"\nref_text = (work / "reference.txt").read_text().strip()\nrepo, revision = "SWivid/F5-TTS", "84e5a410d9cead4de2f847e7c9369a6440bdfaca"\nprint("Downloading F5-TTS v1 Base (pinned).", flush=True)\nstarted = perf_counter()\ncheckpoint = hf_hub_download(repo, "F5TTS_v1_Base/model_1250000.safetensors", revision=revision)\nvocab = hf_hub_download(repo, "F5TTS_v1_Base/vocab.txt", revision=revision)\ndownload_seconds = perf_counter() - started\nstarted = perf_counter()\nmodel = F5TTS(ckpt_file=checkpoint, vocab_file=vocab, device="cuda:0")\ntorch.cuda.synchronize()\nload_seconds = perf_counter() - started\ndtype = str(next(model.ema_model.parameters()).dtype)\nprint("F5-TTS ready:", dtype, flush=True)\n\nrecord = {\n    "label": "AI persona simulation, experimental reference voice",\n    "model": "F5TTS_v1_Base", "repo_revision": revision, "package_version": version("f5-tts"),\n    "gpu": torch.cuda.get_device_name(0), "device": "cuda:0", "dtype": dtype, "torch": torch.__version__,\n    "download_seconds": round(download_seconds, 3), "load_seconds": round(load_seconds, 3),\n    "reference_sha256": hashlib.sha256(reference.read_bytes()).hexdigest(),\n    "answer_sha256": hashlib.sha256(fixture["text"].encode()).hexdigest(), "runs": [],\n}\norder = [(32, "warmup"), (32, "a"), (16, "a"), (16, "b"), (32, "b")]\nfor steps, trial in order:\n    started = perf_counter()\n    messages = []\n    with redirect_stdout(io.StringIO()):\n        wav, rate, _ = model.infer(ref_file=str(reference), ref_text=ref_text,\n            gen_text=fixture["text"], nfe_step=steps, seed=42, show_info=messages.append, progress=None)\n    torch.cuda.synchronize()\n    seconds = perf_counter() - started\n    assert wav is not None and np.isfinite(wav).all() and rate == 24000 and wav.ndim == 1\n    duration = len(wav) / rate\n    assert 2 < duration < 60\n    name = f"experimental-ai-voice-f5-{steps}-{trial}.wav"\n    path = out / name\n    sf.write(path, wav, rate, subtype="PCM_16")\n    record["runs"].append({\n        "steps": steps, "trial": trial, "seconds": round(seconds, 3), "duration_seconds": round(duration, 3),\n        "rtf": round(seconds/duration, 4), "wav": name, "sha256": hashlib.sha256(path.read_bytes()).hexdigest(),\n        "batches": next((info for info in messages if "batches" in info), None),\n    })\n    (out/"result-f5.json").write_text(json.dumps(record, indent=2) + "\\n")\n    print(f"F5 {steps} steps {trial}: {seconds:.3f}s for {duration:.3f}s of audio", flush=True)\nrecord["warm_median_seconds"] = {\n    str(steps): statistics.median(r["seconds"] for r in record["runs"] if r["steps"]==steps and r["trial"]!="warmup")\n    for steps in (16,32)\n}\n(out/"result-f5.json").write_text(json.dumps(record, indent=2) + "\\n")\n')
subprocess.run([sys.executable, str(runner), str(work), str(out)], timeout=700, check=True)
